In [ ]:
!pip install pandas

In [76]:
import pandas
import re

year = 2020

In [ ]:
data = pandas.read_csv(f"data/{year}/{year}.csv")
data.info()

In [78]:
def normalizar_classe(s):
    if pandas.isna(s):
        return "NÃO INFORMADO"
    s = s.upper().strip()
    if re.search(r"OBJETO LAN.ADO", s):
        return "OBJETO LANÇADO CONTRA O VEÍCULO"
    return s

data["CLASSE"] = data["CLASSE"].apply(normalizar_classe)

In [79]:
def normalizar_subclasse(s):
    if pandas.isna(s):
        return "NÃO INFORMADO"
    s = s.upper().strip()
    
    regras = [
        (r"DEFENSA|BARREIRA|SUBMARINO", "CHOQUE-DEFENSA/BARREIRA"),
        (r"DRENAGEM", "CHOQUE-ELEMENTO DE DRENAGEM"),
        (r"TALUDE|BARRANCO|CORTE", "CHOQUE-TALUDE/BARRANCO"),
        (r"MEIO.?FIO|CALÇAMENTO", "CHOQUE-MEIO FIO"),
        (r"ÁRVORE|ARVORE", "CHOQUE-ÁRVORE"),
        (r"POSTE", "CHOQUE-POSTE"),
        (r"BURACO", "CHOQUE-BURACO"),
        (r"OBJETO.*PISTA|OBJETO SOBRE A VIA|VEÍCULO PARADO NA PISTA", "CHOQUE-OBJETO NA PISTA"),
        (r"VEÍCULO PARADO NO ACOSTAMENTO", "CHOQUE-VEÍCULO PARADO NO ACOSTAMENTO"),
        (r"OAE|PILAR|VIADUTO|PONTE", "CHOQUE-OAE (PONTE/VIADUTO)"),
        (r"PRAÇA|CABINE|CANCELA|PEDÁGIO", "CHOQUE-PRAÇA DE PEDÁGIO"),
        (r"SINALIZAÇÃO|EQUIPAMENTO|PAINEL", "CHOQUE-SINALIZAÇÃO/EQUIPAMENTO"),
        (r"CERCA|ALAMBRADO|MOURÃO", "CHOQUE-CERCAS/ALAMBRADOS"),
        (r"EDIFICAÇÃO|ILHA|MATACÃO|OUTROS|NÃO IDENTIF", "CHOQUE-OUTROS"),
        
        (r"^FRONTAL$|COLIS.O-FRONTAL", "COLISÃO-FRONTAL"),
        (r"^TRASEIRA$|COLIS.O-TRASEIRA", "COLISÃO-TRASEIRA"),
        (r"^LATERAL$|COLIS.O-LATERAL", "COLISÃO-LATERAL"),
        (r"^TRANSVERSAL$|COLIS.O-TRANSVERSAL", "COLISÃO-TRANSVERSAL"),
        
        (r"^TOMBAMENTO$", "TOMBAMENTO"),
        (r"TOMBAMENTO-MOTO", "TOMBAMENTO-MOTO"),
        (r"TOMBAMENTO.*PESAD", "TOMBAMENTO-VEÍCULO PESADO"),
        (r"TOMBAMENTO-BICICLETA", "TOMBAMENTO-BICICLETA"),
        
        (r"^CAPOTAMENTO$", "CAPOTAMENTO"),
        (r"^ENGAVETAMENTO$", "ENGAVETAMENTO"),
        
        (r"SUICID|SUICÍD", "ATROP. PEDESTRE-SUICÍDIO"),
        (r"CICLISTA", "ATROP. PEDESTRE-CICLISTA"),
        (r"ATROP\.? PEDESTRE|DE PEDESTRE|PEDESTRE USUÁRIO", "ATROP. PEDESTRE-OUTROS"),
        
        (r"ANIMAL.*SILVESTRE.*GRANDE", "ATROP. ANIMAL-SILVESTRE GRANDE"),
        (r"ANIMAL.*SILVESTRE.*M.DIO", "ATROP. ANIMAL-SILVESTRE MÉDIO"),
        (r"ANIMAL.*SILVESTRE.*PEQUENO", "ATROP. ANIMAL-SILVESTRE PEQUENO"),
        (r"ANIMAL.*DOM.STICO.*GRANDE", "ATROP. ANIMAL-DOMÉSTICO GRANDE"),
        (r"ANIMAL.*DOM.STICO.*M.DIO", "ATROP. ANIMAL-DOMÉSTICO MÉDIO"),
        (r"ANIMAL.*DOM.STICO.*PEQUENO", "ATROP. ANIMAL-DOMÉSTICO PEQUENO"),
        (r"ANIMAL", "ATROP. ANIMAL-OUTROS"),
        
        (r"^QUEDA-MOTO$", "QUEDA-MOTO"),
        (r"^QUEDA-CICLISTA$", "QUEDA-CICLISTA"),
        (r"RIBANCEIRA|EM RIBANCEIRA", "QUEDA-RIBANCEIRA/OAE"),
        (r"QUEDA-CARGA", "QUEDA-CARGA"),
        (r"^QUEDA$|TABLUDE", "QUEDA-OUTROS"),
        
        (r"LAN.ADO", "OBJETO LANÇADO CONTRA O VEÍCULO"),
        (r"INC.NDIO", "INCÊNDIO"),
        (r"SA.DA DE PISTA", "SAÍDA DE PISTA"),
    ]
    
    for padrao, categoria in regras:
        if re.search(padrao, s):
            return categoria
    
    return "OUTROS/NÃO CLASSIFICADO"

data["SUBCLASSE"] = data["SUBCLASSE"].apply(normalizar_subclasse)

In [75]:
data = data.sort_values(by=[data.DATA.name, data.HORA.name], ascending=True)

data.loc[(data.RODOVIA.str.contains("330")), :].drop(columns=["_id","VITIMAS_SEM_INFO","VISIBILIDADE","CONDICAO_METERIOLOGICA"]).to_csv(f"data/{year}/p{year}.csv", index=False)

In [ ]:
print(data.head())